In [2]:
from pathlib import Path
from eeg_music.data import EEGMusicDataset, MappedDataset, MelParams, TrialData, WavRAW, prepare_trial, wavraw_to_melspectrogram
from eeg_music.onset_odf_detection import trial_to_odf
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from numpy.typing import NDArray

from eeg_music.data import EEGMusicDataset, MappedDataset
from eeg_music.data import notch_filter_trial, filter_trial, rereference_trial, std_normalize_trial_channels
from eeg_music.ica_analysis import band_power_trial
from eeg_music.ica_analysis import ica_clean_trial_hydrocel129
from eeg_music.ica_analysis import (
    apply_ica,
    clean_ica_artifacts,
    ica_band_power_trial,
    ica_band_power_trial_1020,
    windowed_band_power,
    _prepare_raw_1020,
)


/home/zmrocze/studia/uwr/masters/eeg-code/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


In [34]:
def resample_wav_only(trial: TrialData, wav_resample: int) -> TrialData:
    """Resample only the WAV data, leaving EEG untouched."""
    music = trial.music_data.get_music()
    if isinstance(music, WavRAW) and music.sample_rate != wav_resample:
        music_resampled = music.resampled(new_sr=wav_resample)
    else:
        music_resampled = music
    
    return TrialData(
        dataset=trial.dataset,
        subject=trial.subject,
        session=trial.session,
        run=trial.run,
        trial_id=trial.trial_id,
        music_filename=trial.music_filename,
        eeg_data=trial.eeg_data,  # Unchanged
        music_data=music_resampled,
    )

def trim_trial_to_match(trial: TrialData, eeg_sr: int = 256, odf_sr: int = 64) -> TrialData:
    """Trim EEG and ODF arrays to matching duration based on their sample rates.
    
    Ensures exact ratio between EEG and ODF samples (eeg_sr / odf_sr).
    
    Args:
        trial: TrialData with EEG and ODF (stored as MelRaw)
        eeg_sr: EEG sample rate (default: 256 Hz)
        odf_sr: ODF sample rate (default: 64 Hz)
    
    Returns:
        TrialData with trimmed EEG and ODF to matching duration
    """
    from eeg_music.data import ArrayEeg, MelRaw
    
    eeg_data = trial.eeg_data.get_array()
    odf_data = trial.music_data.get_music()
    
    eeg_samples = eeg_data.data.shape[1]
    odf_frames = odf_data.mel.shape[1]
    
    # Calculate minimum length ensuring exact ratio
    ratio = eeg_sr // odf_sr  # Should be exactly 4 for 256/64
    
    # Find the minimum in ODF frames, then scale to EEG
    max_odf_frames = min(odf_frames, eeg_samples // ratio)
    new_odf_frames = max_odf_frames
    new_eeg_samples = max_odf_frames * ratio
    
    trimmed_eeg = eeg_data.data[:, :new_eeg_samples]
    trimmed_odf = odf_data.mel[:, :new_odf_frames]
    
    return TrialData(
        dataset=trial.dataset,
        subject=trial.subject,
        session=trial.session,
        run=trial.run,
        trial_id=trial.trial_id,
        music_filename=trial.music_filename,
        eeg_data=ArrayEeg(data=trimmed_eeg, ch_names=eeg_data.ch_names, sfreq=eeg_data.sfreq),
        music_data=MelRaw(
            mel=trimmed_odf,
            sample_rate=odf_data.sample_rate,
            hop_length=odf_data.hop_length,
            fmin=odf_data.fmin,
            fmax=odf_data.fmax,
            to_db=odf_data.to_db,
        ),
    )

def calculate_odf_min_max(dataset: MappedDataset) -> tuple[float, float]:
    """Calculate global min and max ODF values across entire dataset."""
    global_min = float('inf')
    global_max = float('-inf')
    
    for trial in dataset:
        odf_data = trial.music_data.get_music()
        odf_values = odf_data.mel
        
        trial_min = odf_values.min()
        trial_max = odf_values.max()
        
        global_min = min(global_min, trial_min)
        global_max = max(global_max, trial_max)
    
    return global_min, global_max

def normalize_odf_trial(trial: TrialData, odf_min: float, odf_max: float) -> TrialData:
    """Normalize ODF values to 0-1 range using (x - min) / (max - min) with global min/max."""
    from eeg_music.data import MelRaw
    
    odf_data = trial.music_data.get_music()
    odf_values = odf_data.mel
    
    if odf_max - odf_min > 0:
        normalized_odf = (odf_values - odf_min) / (odf_max - odf_min)
    else:
        normalized_odf = odf_values
    
    return TrialData(
        dataset=trial.dataset,
        subject=trial.subject,
        session=trial.session,
        run=trial.run,
        trial_id=trial.trial_id,
        music_filename=trial.music_filename,
        eeg_data=trial.eeg_data,
        music_data=MelRaw(
            mel=normalized_odf,
            sample_rate=odf_data.sample_rate,
            hop_length=odf_data.hop_length,
            fmin=odf_data.fmin,
            fmax=odf_data.fmax,
            to_db=odf_data.to_db,
        ),
    )

ds = EEGMusicDataset.load_ondisk(
    Path("./datasets/musing_g_export_4_wav")
  )

mds = MappedDataset(ds, lambda t: 
  trial_to_odf(resample_wav_only(t, wav_resample=32768)))

In [36]:

mds_ica = MappedDataset(ds, lambda x: 
  normalize_odf_trial(trim_trial_to_match(
    trial_to_odf(
        ica_clean_trial_hydrocel129(
          rereference_trial(
            prepare_trial(
              notch_filter_trial(x, 50.0),
              eeg_resample=256,
              eeg_l_freq=0.5,
              eeg_h_freq=50.0,
              wav_resample=32768,
            )
          ), 
          return_n_components=12
        ), 
      ),
    eeg_sr=256,
    odf_sr=64
  ), odf_min, odf_max)
)
x = mds_ica[0]
x.eeg_data.get_array().data.shape, x.music_data.mel.shape

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 246 of 35071 (0.70%) samples, retaining 34825 (99.30%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

((12, 35068), (1, 8767))

In [35]:
odf_min, odf_max = calculate_odf_min_max(mds_ica)
print(f"ODF range: [{odf_min:.4f}, {odf_max:.4f}]")

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 246 of 35071 (0.70%) samples, retaining 34825 (99.30%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 21 of 31999 (0.07%) samples, retaining 31978 (99.93%) samples.
Selecting by number: 22 components
Fitting ICA took 3.3s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 54 of 36607 (0.15%) samples, retaining 36553 (99.85%) samples.
Selecting by number: 22 components
Fitting ICA took 0.6s.


...

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 6601 samples (6.601 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 217 of 31743 (0.68%) samples, retaining 31526 (99.32%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 6601 samples (6.601 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 46 of 32767 (0.14%) samples, retaining 32721 (99.86%) samples.
Selecting by number: 22 components
Fitting ICA took 0.3s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

ODF range: [0.0000, 118703.1406]


In [37]:

mds_ica.save(Path("./datasets/musing_preprocessed/musing_rawica_odf"))

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 246 of 35071 (0.70%) samples, retaining 34825 (99.30%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 21 of 31999 (0.07%) samples, retaining 31978 (99.93%) samples.
Selecting by number: 22 components
Fitting ICA took 0.7s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 54 of 36607 (0.15%) samples, retaining 36553 (99.85%) samples.
Selecting by number: 22 components
Fitting ICA took 0.7s.


...

/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 6601 samples (6.601 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 217 of 31743 (0.68%) samples, retaining 31526 (99.32%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 6601 samples (6.601 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 46 of 32767 (0.14%) samples, retaining 32721 (99.86%) samples.
Selecting by number: 22 components
Fitting ICA took 0.3s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

In [38]:
mds_noica = MappedDataset(ds, lambda x: 
  normalize_odf_trial(trim_trial_to_match(
    trial_to_odf(
      rereference_trial(prepare_trial(
          notch_filter_trial(x, 50.0), 
          eeg_l_freq=0.5, 
          eeg_h_freq=50.0,
          wav_resample=32768,
          pick_channels=picked,
          eeg_resample=256,
      ))
    ),
    eeg_sr=256,
    odf_sr=64
  ),
  odf_min, odf_max)
)
x = mds_noica[0]
x.eeg_data.get_array().data.shape, x.music_data.mel.shape

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)



((12, 35068), (1, 8767))

In [39]:
mds_noica.save(Path("./datasets/musing_preprocessed/musing_rawnoica_odf"))

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband ed

In [32]:
mds_ica[0].music_data.mel

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 1651 samples (6.604 s)

Fitting ICA to data using 129 channels (please be patient, this may take a while)
Omitting 246 of 35071 (0.70%) samples, retaining 34825 (99.30%) samples.
Selecting by number: 22 components
Fitting ICA took 0.4s.


/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  component_labels = label_components(raw, ica, method="iclabel")["labels"]
/home/zmrocze/studia/uwr/masters/eeg-code/src/eeg_music/ica_analysis.py:61: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designe

array([[   0.    ,    0.    ,    0.    , ..., 1341.8792, 1583.594 ,
        4240.2827]], dtype=float32)

In [19]:
ds[0].music_data.get_music().sample_rate

44100

In [20]:
x = mds[0].music_data
x.mel.shape

(1, 29008)

In [14]:
ds.save(Path("./datasets/musing_preprocessed/musing_wav_60ch_2/"))

Overwriting existing metadata at datasets/musing_preprocessed/musing_wav_60ch_2/metadata.json


In [21]:
mds.save(Path("./datasets/musing_preprocessed/musing_odf_60ch/"))

In [ ]:
# prepare 2 datasets using ica and no ica
#   and resample and calculate odf
#
#
eeg = mds[0].eeg_data.get_array()

In [28]:
eeg.data.shape

(60, 1351)